In [7]:
import os
import csv
import json
import sqlite3
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

In [8]:
RAW       = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)


In [9]:
print("\n" + "="*60)
print("  SECTION 1 — FILE INGESTION")
print("="*60)


  SECTION 1 — FILE INGESTION


In [10]:
customers = [
    ["customer_id", "name",           "email",              "city",        "tier"  ],
    [1,             "Alice Johnson",  "alice@email.com",    "New York",    "Gold"  ],
    [2,             "Bob Smith",      "bob@email.com",      "Chicago",     "Silver"],
    [3,             "Carol White",    "carol@email.com",    "Los Angeles", "Gold"  ],
    [4,             "David Lee",      "david@email.com",    "Houston",     "Bronze"],
    [5,             "Eva Brown",      "eva@email.com",      "Phoenix",     "Silver"],
]

In [11]:
with open(RAW / "customers.csv", "w", newline="") as f:
    csv.writer(f).writerows(customers)
print("Created: customers.csv")

Created: customers.csv


In [12]:
orders = [
    {"order_id": 101, "customer_id": 1, "product": "Laptop",  "qty": 1, "price": 999.99, "status": "delivered"},
    {"order_id": 102, "customer_id": 2, "product": "Mouse",   "qty": 2, "price":  25.00, "status": "shipped"  },
    {"order_id": 103, "customer_id": 3, "product": "Monitor", "qty": 1, "price": 349.99, "status": "pending"  },
    {"order_id": 104, "customer_id": 1, "product": "Keyboard","qty": 1, "price":  79.99, "status": "delivered"},
    {"order_id": 105, "customer_id": 5, "product": "Webcam",  "qty": 1, "price":  89.99, "status": "shipped"  },
]

In [13]:
with open(RAW / "orders.json", "w") as f:
    json.dump(orders, f, indent=2)
print("Created: orders.json")

Created: orders.json


In [14]:
products_df   = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "name":       ["Laptop", "Mouse", "Monitor", "Keyboard", "Webcam"],
    "price":      [999.99, 25.00, 349.99, 79.99, 89.99],
    "stock":      [50, 200, 75, 150, 120],
})

In [15]:
categories_df = pd.DataFrame({
    "category_id":   [1],
    "category_name": ["Electronics"],
})

In [17]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
with pd.ExcelWriter(RAW / "products.xlsx", engine="openpyxl") as writer:
    products_df.to_excel(writer,   sheet_name="Products",   index=False)
    categories_df.to_excel(writer, sheet_name="Categories", index=False)
print("Created: products.xlsx  (2 sheets)")

Created: products.xlsx  (2 sheets)


In [19]:
print("\n--- Reading customers.csv ---")
df_customers = pd.read_csv(RAW / "customers.csv")
print(df_customers)


--- Reading customers.csv ---
   customer_id           name            email         city    tier
0            1  Alice Johnson  alice@email.com     New York    Gold
1            2      Bob Smith    bob@email.com      Chicago  Silver
2            3    Carol White  carol@email.com  Los Angeles    Gold
3            4      David Lee  david@email.com      Houston  Bronze
4            5      Eva Brown    eva@email.com      Phoenix  Silver


In [20]:
print("\n--- Reading orders.json ---")
with open(RAW / "orders.json") as f:
    raw = json.load(f)
df_orders = pd.DataFrame(raw)
print(df_orders)



--- Reading orders.json ---
   order_id  customer_id   product  qty   price     status
0       101            1    Laptop    1  999.99  delivered
1       102            2     Mouse    2   25.00    shipped
2       103            3   Monitor    1  349.99    pending
3       104            1  Keyboard    1   79.99  delivered
4       105            5    Webcam    1   89.99    shipped


In [21]:
# Read Excel — sheet by name
print("\n--- Reading products.xlsx  (sheet=Products) ---")
df_products = pd.read_excel(RAW / "products.xlsx", sheet_name="Products")
print(df_products)


--- Reading products.xlsx  (sheet=Products) ---
   product_id      name   price  stock
0           1    Laptop  999.99     50
1           2     Mouse   25.00    200
2           3   Monitor  349.99     75
3           4  Keyboard   79.99    150
4           5    Webcam   89.99    120


In [22]:
print("\n--- Reading products.xlsx  (sheet=Categories) ---")
df_categories = pd.read_excel(RAW / "products.xlsx", sheet_name="Categories")
print(df_categories)


--- Reading products.xlsx  (sheet=Categories) ---
   category_id category_name
0            1   Electronics


In [23]:
print("\n" + "="*60)
print("  SECTION 2 — API INGESTION")
print("  https://jsonplaceholder.typicode.com")
print("="*60)


  SECTION 2 — API INGESTION
  https://jsonplaceholder.typicode.com


In [24]:
BASE_URL = "https://jsonplaceholder.typicode.com"

In [25]:
def api_get(endpoint, params=None):
    url = BASE_URL + endpoint
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            if attempt == 2:
                raise
    return []

In [27]:
SAMPLE_USERS = [
    {"id":1,"name":"Leanne Graham","username":"Bret","email":"Sincere@april.biz",
     "phone":"1-770-736-0988","address":{"city":"Gwenborough"},"company":{"name":"Romaguera-Crona"}},
    {"id":2,"name":"Ervin Howell","username":"Antonette","email":"Shanna@melissa.tv",
     "phone":"010-692-6593","address":{"city":"Wisokyburgh"},"company":{"name":"Deckow-Crist"}},
    {"id":3,"name":"Clementine Bauch","username":"Samantha","email":"Nathan@yesenia.net",
     "phone":"1-463-123-4447","address":{"city":"McKenziehaven"},"company":{"name":"Romaguera-Jacobson"}},
    {"id":4,"name":"Patricia Lebsack","username":"Karianne","email":"Julianne.OConner@kory.org",
     "phone":"493-170-9623","address":{"city":"South Elvis"},"company":{"name":"Robel-Corkery"}},
    {"id":5,"name":"Chelsey Dietrich","username":"Kamren","email":"Lucio_Hettinger@annie.ca",
     "phone":"(254)954-1289","address":{"city":"Roscoeview"},"company":{"name":"Keebler LLC"}},
]

In [28]:
SAMPLE_POSTS = [
    {"userId":1,"id":1,"title":"sunt aut facere repellat","body":"quia et suscipit recusandae"},
    {"userId":1,"id":2,"title":"qui est esse","body":"est rerum tempore vitae sequi"},
    {"userId":2,"id":3,"title":"ea molestias quasi exercitationem","body":"et iusto sed quo iure"},
    {"userId":2,"id":4,"title":"eum et est occaecati","body":"ullam et saepe reiciendis"},
    {"userId":3,"id":5,"title":"nesciunt quas odio","body":"repudiandae veniam quaerat"},
]

In [29]:
SAMPLE_COMMENTS = [
    {"postId":1,"id":1,"name":"id labore ex et quam laborum","email":"Eliseo@gardner.biz","body":"laudantium enim quasi est"},
    {"postId":1,"id":2,"name":"quo vero reiciendis velit similique earum","email":"Jayne_Kuhic@sydney.com","body":"est natus enim nihil est dolore"},
    {"postId":1,"id":3,"name":"odio adipisci rerum aut animi","email":"Nikita@garfield.biz","body":"quia molestiae reprehenderit quasi"},
]

In [30]:
print("\n--- Fetching /users ---")


--- Fetching /users ---


In [31]:
try:
    users_raw = api_get("/users")
    print("  (live API)")
except Exception:
    users_raw = SAMPLE_USERS
    print("  (network unavailable — using sample data)")


  (live API)


In [32]:
users_raw

[{'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'address': {'street': 'Kulas Light',
   'suite': 'Apt. 556',
   'city': 'Gwenborough',
   'zipcode': '92998-3874',
   'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
  'phone': '1-770-736-8031 x56442',
  'website': 'hildegard.org',
  'company': {'name': 'Romaguera-Crona',
   'catchPhrase': 'Multi-layered client-server neural-net',
   'bs': 'harness real-time e-markets'}},
 {'id': 2,
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'email': 'Shanna@melissa.tv',
  'address': {'street': 'Victor Plains',
   'suite': 'Suite 879',
   'city': 'Wisokyburgh',
   'zipcode': '90566-7771',
   'geo': {'lat': '-43.9509', 'lng': '-34.4618'}},
  'phone': '010-692-6593 x09125',
  'website': 'anastasia.net',
  'company': {'name': 'Deckow-Crist',
   'catchPhrase': 'Proactive didactic contingency',
   'bs': 'synergize scalable supply-chains'}},
 {'id': 3,
  'name': 'Clementine Bauch',
  'username': 'Samantha

In [36]:
df_users = pd.json_normalize(users_raw)

In [40]:
users_raw

[{'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'address': {'street': 'Kulas Light',
   'suite': 'Apt. 556',
   'city': 'Gwenborough',
   'zipcode': '92998-3874',
   'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
  'phone': '1-770-736-8031 x56442',
  'website': 'hildegard.org',
  'company': {'name': 'Romaguera-Crona',
   'catchPhrase': 'Multi-layered client-server neural-net',
   'bs': 'harness real-time e-markets'}},
 {'id': 2,
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'email': 'Shanna@melissa.tv',
  'address': {'street': 'Victor Plains',
   'suite': 'Suite 879',
   'city': 'Wisokyburgh',
   'zipcode': '90566-7771',
   'geo': {'lat': '-43.9509', 'lng': '-34.4618'}},
  'phone': '010-692-6593 x09125',
  'website': 'anastasia.net',
  'company': {'name': 'Deckow-Crist',
   'catchPhrase': 'Proactive didactic contingency',
   'bs': 'synergize scalable supply-chains'}},
 {'id': 3,
  'name': 'Clementine Bauch',
  'username': 'Samantha

In [37]:
df_users

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,1-477-935-8478 x6430,ola.org,Norberto Crossing,Apt. 950,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,e-enable innovative applications
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,210.067.6132,elvis.io,Rex Trail,Suite 280,Howemouth,58804-1099,24.8918,21.8984,Johns Group,Configurable multimedia task-force,generate enterprise e-tailers
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Ellsworth Summit,Suite 729,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,Implemented secondary concept,e-enable extensible e-tailers
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,aggregate real-time technologies
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Kattie Turnpike,Suite 198,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,Centralized empowering task-force,target end-to-end models


In [34]:
keep = ["id", "name", "username", "email", "phone", "address.city", "company.name"]
df_users = df_users[[c for c in keep if c in df_users.columns]]
df_users.columns = [c.replace("address.", "").replace("company.", "") for c in df_users.columns]
print(df_users.to_string(index=False))

 id                     name         username                     email                 phone           city               name
  1            Leanne Graham             Bret         Sincere@april.biz 1-770-736-8031 x56442    Gwenborough    Romaguera-Crona
  2             Ervin Howell        Antonette         Shanna@melissa.tv   010-692-6593 x09125    Wisokyburgh       Deckow-Crist
  3         Clementine Bauch         Samantha        Nathan@yesenia.net        1-463-123-4447  McKenziehaven Romaguera-Jacobson
  4         Patricia Lebsack         Karianne Julianne.OConner@kory.org     493-170-9623 x156    South Elvis      Robel-Corkery
  5         Chelsey Dietrich           Kamren  Lucio_Hettinger@annie.ca         (254)954-1289     Roscoeview        Keebler LLC
  6     Mrs. Dennis Schulist Leopoldo_Corkery   Karley_Dach@jasper.info  1-477-935-8478 x6430  South Christy  Considine-Lockman
  7          Kurtis Weissnat     Elwyn.Skiles    Telly.Hoeger@billy.biz          210.067.6132      Howem

In [38]:
print("\n--- Fetching /posts  (paginated, 10 per page) ---")

all_posts = []
page = 1
page_size = 10



--- Fetching /posts  (paginated, 10 per page) ---


In [39]:
while True:
    try:
        data = api_get("/posts", params={"_page": page, "_limit": page_size})
    except Exception:
        # Fallback: simulate 2 pages of 3 records each
        if page == 1:
            data = SAMPLE_POSTS[:3]
        elif page == 2:
            data = SAMPLE_POSTS[3:]
        else:
            data = []

    if not data:           # empty list = no more pages
        break

    all_posts.extend(data)
    print(f"  Page {page} → {len(data)} records  (total so far: {len(all_posts)})")

    if len(data) < page_size:   # last page (partial)
        break

    page += 1

  Page 1 → 10 records  (total so far: 10)
  Page 2 → 10 records  (total so far: 20)
  Page 3 → 10 records  (total so far: 30)
  Page 4 → 10 records  (total so far: 40)
  Page 5 → 10 records  (total so far: 50)
  Page 6 → 10 records  (total so far: 60)
  Page 7 → 10 records  (total so far: 70)
  Page 8 → 10 records  (total so far: 80)
  Page 9 → 10 records  (total so far: 90)
  Page 10 → 10 records  (total so far: 100)


In [41]:
all_posts

[{'userId': 1,
  'id': 1,
  'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
  'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'},
 {'userId': 1,
  'id': 2,
  'title': 'qui est esse',
  'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'},
 {'userId': 1,
  'id': 3,
  'title': 'ea molestias quasi exercitationem repellat qui ipsa sit aut',
  'body': 'et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut'},
 {'userId': 1,
  'id': 4,
  'title': 'eum et est occaecati',
  'body': 'ullam et saepe reiciendis voluptatem adipisci\nsit amet autem assumenda provid

In [42]:
df_posts = pd.DataFrame(all_posts)

In [43]:
df_posts

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...
...,...,...,...,...
95,10,96,quaerat velit veniam amet cupiditate aut numqu...,in non odio excepturi sint eum\nlabore volupta...
96,10,97,quas fugiat ut perspiciatis vero provident,eum non blanditiis soluta porro quibusdam volu...
97,10,98,laboriosam dolor voluptates,doloremque ex facilis sit sint culpa\nsoluta a...
98,10,99,temporibus sit alias delectus eligendi possimu...,quo deleniti praesentium dicta non quod\naut e...


In [44]:
print(f"\nTotal posts fetched: {len(df_posts)}")
print(df_posts.head(3).to_string(index=False))


Total posts fetched: 100
 userId  id                                                                      title                                                                                                                                                                                                              body
      1   1 sunt aut facere repellat provident occaecati excepturi optio reprehenderit                                                 quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto
      1   2                                                               qui est esse est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla
      1   3                ea molestias quasi exercitationem repellat qui ipsa sit 

In [45]:
# --------------------------------------------------------------------------
# 2C. Fetch related resource  (/comments for a specific post)
# --------------------------------------------------------------------------

In [46]:
print("\n--- Fetching comments for post_id=1 ---")
try:
    comments_raw = api_get("/comments", params={"postId": 1})
    print("  (live API)")
except Exception:
    comments_raw = SAMPLE_COMMENTS
    print("  (network unavailable — using sample data)")

df_comments = pd.DataFrame(comments_raw)
print(df_comments[["id", "name", "email", "body"]].to_string(index=False))


--- Fetching comments for post_id=1 ---
  (live API)
 id                                      name                  email                                                                                                                                                                                                                          body
  1              id labore ex et quam laborum     Eliseo@gardner.biz                                                                       laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium
  2 quo vero reiciendis velit similique earum Jayne_Kuhic@sydney.com                                                   est natus enim nihil est dolore omnis voluptatem numquam\net omnis occaecati quod ullam at\nvoluptatem error expedita pariatur\nnihil sint nostrum voluptatem reiciendis et
  3             odio adipisci rerum aut animi    Nikita@garfield.biz quia

In [47]:
print("\n" + "="*60)
print("  SECTION 3 — DATABASE INGESTION")
print("  (SQLite demo — swap one line for Postgres or Snowflake)")
print("="*60)


  SECTION 3 — DATABASE INGESTION
  (SQLite demo — swap one line for Postgres or Snowflake)


In [48]:
DB_FILE = Path("data/source.db")

In [73]:
conn = sqlite3.connect(DB_FILE)
cur  = conn.cursor()

In [50]:
cur.executescript("""
DROP TABLE IF EXISTS transactions;
DROP TABLE IF EXISTS employees;

CREATE TABLE transactions (
    txn_id      INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product     TEXT,
    amount      REAL,
    status      TEXT,
    created_at  TEXT,
    updated_at  TEXT
);

CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    name       TEXT,
    department TEXT,
    salary     REAL,
    hire_date  TEXT
);
""")

In [51]:
cur.executemany("INSERT INTO transactions VALUES (?,?,?,?,?,?,?)", [
    (1, 1, "Laptop",     999.99, "completed", "2024-01-10", "2024-01-10 09:00"),
    (2, 2, "Keyboard",    79.99, "completed", "2024-01-15", "2024-01-15 14:00"),
    (3, 3, "Monitor",    349.99, "shipped",   "2024-02-01", "2024-02-02 08:00"),
    (4, 1, "Mouse",       25.00, "completed", "2024-02-10", "2024-02-10 16:00"),
    (5, 4, "Tablet",     599.99, "completed", "2024-02-20", "2024-02-20 11:00"),
    (6, 5, "Headphones", 199.99, "completed", "2024-03-01", "2024-03-01 09:00"),
    (7, 2, "USB Hub",     49.99, "pending",   "2024-03-05", "2024-03-05 13:00"),
    (8, 6, "Workstation",1299.99,"completed", "2024-03-10", "2024-03-10 10:00"),
])


In [52]:
cur.executemany("INSERT INTO employees VALUES (?,?,?,?,?)", [
    (101, "Sarah Connor", "Engineering", 95000, "2021-06-01"),
    (102, "John Doe",     "Sales",       72000, "2022-01-15"),
    (103, "Jane Smith",   "HR",          65000, "2020-09-10"),
    (104, "Mike Ross",    "Engineering", 88000, "2023-03-20"),
    (105, "Rachel Zane",  "Legal",       98000, "2019-11-05"),
])

In [53]:
conn.commit()

In [54]:
print("Database seeded: transactions(8 rows) + employees(5 rows)")

Database seeded: transactions(8 rows) + employees(5 rows)


In [55]:
print("\n--- Read full table: transactions ---")
df_txn = pd.read_sql("SELECT * FROM transactions", conn)
print(df_txn.to_string(index=False))



--- Read full table: transactions ---
 txn_id  customer_id     product  amount    status created_at       updated_at
      1            1      Laptop  999.99 completed 2024-01-10 2024-01-10 09:00
      2            2    Keyboard   79.99 completed 2024-01-15 2024-01-15 14:00
      3            3     Monitor  349.99   shipped 2024-02-01 2024-02-02 08:00
      4            1       Mouse   25.00 completed 2024-02-10 2024-02-10 16:00
      5            4      Tablet  599.99 completed 2024-02-20 2024-02-20 11:00
      6            5  Headphones  199.99 completed 2024-03-01 2024-03-01 09:00
      7            2     USB Hub   49.99   pending 2024-03-05 2024-03-05 13:00
      8            6 Workstation 1299.99 completed 2024-03-10 2024-03-10 10:00


In [56]:

print("\n--- Read full table: employees ---")
df_emp = pd.read_sql("SELECT * FROM employees", conn)
print(df_emp.to_string(index=False))


--- Read full table: employees ---
 emp_id         name  department  salary  hire_date
    101 Sarah Connor Engineering 95000.0 2021-06-01
    102     John Doe       Sales 72000.0 2022-01-15
    103   Jane Smith          HR 65000.0 2020-09-10
    104    Mike Ross Engineering 88000.0 2023-03-20
    105  Rachel Zane       Legal 98000.0 2019-11-05


In [57]:
print("\n--- Filter: transactions where amount > 200 ---")
df_big = pd.read_sql(
    "SELECT * FROM transactions WHERE amount > 200 ORDER BY amount DESC",
    conn
)
print(df_big.to_string(index=False))


--- Filter: transactions where amount > 200 ---
 txn_id  customer_id     product  amount    status created_at       updated_at
      8            6 Workstation 1299.99 completed 2024-03-10 2024-03-10 10:00
      1            1      Laptop  999.99 completed 2024-01-10 2024-01-10 09:00
      5            4      Tablet  599.99 completed 2024-02-20 2024-02-20 11:00
      3            3     Monitor  349.99   shipped 2024-02-01 2024-02-02 08:00


In [58]:
print("\n--- Chunked read (chunk_size=3) ---")
offset = 0
chunk_size = 3
chunk_num  = 0


--- Chunked read (chunk_size=3) ---


In [59]:
while True:
    chunk = pd.read_sql(
        f"SELECT * FROM transactions LIMIT {chunk_size} OFFSET {offset}",
        conn
    )
    if chunk.empty:
        break
    chunk_num += 1
    print(f"  Chunk {chunk_num}: {len(chunk)} rows")
    offset += chunk_size

print(f"  Total chunks: {chunk_num}")

  Chunk 1: 3 rows
  Chunk 2: 3 rows
  Chunk 3: 2 rows
  Total chunks: 3


In [62]:
# import psycopg2
# conn = psycopg2.connect(
#   host="localhost", port=5432,
#   dbname="test", user="postgres", password="1234"
# )

In [64]:
# import snowflake.connector
# conn = snowflake.connector.connect(
#     user='detrainings01'
#     , password='DataEngineering@2026'
#     , account='sqqpjbk-pi02317'
#     , warehouse='COMPUTE_WH'
#     , role='SYSADMIN'
#     , database='MY_DB1'
#     , schema='RAW'
# )

20:41:34  [INFO    ]  Snowflake Connector for Python Version: 4.2.0, Python Version: 3.11.9, Platform: Windows-10-10.0.26200-SP0
20:41:34  [INFO    ]  Connecting to GLOBAL Snowflake domain
20:41:34  [INFO    ]  Found credentials in shared credentials file: ~/.aws/credentials


In [65]:
print("\n" + "="*60)
print("  SECTION 4 — INCREMENTAL vs FULL LOAD")
print("="*60)



  SECTION 4 — INCREMENTAL vs FULL LOAD


In [66]:
WATERMARK_FILE = Path("data/watermark.json")

In [67]:
def load_watermark(table_name):
    """Read the saved watermark for a table. Returns None if first run."""
    if not WATERMARK_FILE.exists():
        return None
    data = json.loads(WATERMARK_FILE.read_text())
    return data.get(table_name)

In [68]:
def save_watermark(table_name, value):
    """Save the watermark after a successful load."""
    data = {}
    if WATERMARK_FILE.exists():
        data = json.loads(WATERMARK_FILE.read_text())
    data[table_name] = str(value)
    WATERMARK_FILE.write_text(json.dumps(data, indent=2))
    print(f"  Watermark saved: {table_name} = {value}")

In [70]:
def full_load(connection, table_name):
    """Read every row from the table."""
    print(f"\n[FULL LOAD] Reading all rows from '{table_name}' ...")
    df = pd.read_sql(f"SELECT * FROM {table_name}", connection)
    print(f"  Loaded: {len(df)} rows")
    print(df.to_string(index=False))
    # Save the latest timestamp as the new watermark
    if "updated_at" in df.columns:
        save_watermark(table_name, df["updated_at"].max())
    return df

In [71]:
def incremental_load(connection, table_name, timestamp_col="updated_at"):
    """Read only rows newer than the last watermark."""
    last_ts = load_watermark(table_name)

    if last_ts is None:
        print(f"\n[INCREMENTAL] No watermark found → running full load instead.")
        return full_load(connection, table_name)

    print(f"\n[INCREMENTAL] Loading '{table_name}' where {timestamp_col} > '{last_ts}' ...")
    df = pd.read_sql(
        f"SELECT * FROM {table_name} WHERE {timestamp_col} > '{last_ts}'",
        connection
    )

    if df.empty:
        print("  No new rows — already up to date.")
        return df

    print(f"  Loaded: {len(df)} new rows")
    print(df.to_string(index=False))
    save_watermark(table_name, df[timestamp_col].max())
    return df

In [72]:
if WATERMARK_FILE.exists():
    WATERMARK_FILE.unlink()

print("""
How it works:
  Run 1 → no watermark → full load   → saves watermark = last updated_at
  Run 2 → same data    → 0 new rows  (nothing changed)
  Run 3 → new rows     → only new rows loaded, watermark advances
  Run 4 → force full   → all rows loaded again
""")


How it works:
  Run 1 → no watermark → full load   → saves watermark = last updated_at
  Run 2 → same data    → 0 new rows  (nothing changed)
  Run 3 → new rows     → only new rows loaded, watermark advances
  Run 4 → force full   → all rows loaded again



In [74]:
incremental_load(conn, "transactions")


[INCREMENTAL] No watermark found → running full load instead.

[FULL LOAD] Reading all rows from 'transactions' ...
  Loaded: 8 rows
 txn_id  customer_id     product  amount    status created_at       updated_at
      1            1      Laptop  999.99 completed 2024-01-10 2024-01-10 09:00
      2            2    Keyboard   79.99 completed 2024-01-15 2024-01-15 14:00
      3            3     Monitor  349.99   shipped 2024-02-01 2024-02-02 08:00
      4            1       Mouse   25.00 completed 2024-02-10 2024-02-10 16:00
      5            4      Tablet  599.99 completed 2024-02-20 2024-02-20 11:00
      6            5  Headphones  199.99 completed 2024-03-01 2024-03-01 09:00
      7            2     USB Hub   49.99   pending 2024-03-05 2024-03-05 13:00
      8            6 Workstation 1299.99 completed 2024-03-10 2024-03-10 10:00
  Watermark saved: transactions = 2024-03-10 10:00


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00


In [75]:
# Run 2 — nothing new
print("─" * 40)
print("RUN 2 — Re-run immediately (no new data)")
incremental_load(conn, "transactions")

────────────────────────────────────────
RUN 2 — Re-run immediately (no new data)

[INCREMENTAL] Loading 'transactions' where updated_at > '2024-03-10 10:00' ...
  No new rows — already up to date.


,txn_id,customer_id,product,amount,status,created_at,updated_at


In [77]:
print("─" * 40)
print("RUN 3 — Adding 3 new rows then running incremental")
conn.executemany("INSERT INTO transactions VALUES (?,?,?,?,?,?,?)", [
    (9,  7, "Printer",   459.99, "completed", "2024-04-01", "2024-04-01 08:00"),
    (10, 8, "Speaker",   129.99, "pending",   "2024-04-02", "2024-04-02 10:00"),
    (11, 3, "MacBook",  2499.99, "completed", "2024-04-05", "2024-04-05 14:00"),
])


────────────────────────────────────────
RUN 3 — Adding 3 new rows then running incremental


IntegrityError: UNIQUE constraint failed: transactions.txn_id

In [78]:
conn.commit()

In [79]:
print("  3 rows inserted.")
incremental_load(conn, "transactions")

  3 rows inserted.

[INCREMENTAL] Loading 'transactions' where updated_at > '2024-03-10 10:00' ...
  Loaded: 3 new rows
 txn_id  customer_id product  amount    status created_at       updated_at
      9            7 Printer  459.99 completed 2024-04-01 2024-04-01 08:00
     10            8 Speaker  129.99   pending 2024-04-02 2024-04-02 10:00
     11            3 MacBook 2499.99 completed 2024-04-05 2024-04-05 14:00
  Watermark saved: transactions = 2024-04-05 14:00


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,9,7,Printer,459.99,completed,2024-04-01,2024-04-01 08:00
1,10,8,Speaker,129.99,pending,2024-04-02,2024-04-02 10:00
2,11,3,MacBook,2499.99,completed,2024-04-05,2024-04-05 14:00


In [80]:
print("─" * 40)
print("RUN 4 — Force full load (reloads everything)")
full_load(conn, "transactions")

────────────────────────────────────────
RUN 4 — Force full load (reloads everything)

[FULL LOAD] Reading all rows from 'transactions' ...
  Loaded: 11 rows
 txn_id  customer_id     product  amount    status created_at       updated_at
      1            1      Laptop  999.99 completed 2024-01-10 2024-01-10 09:00
      2            2    Keyboard   79.99 completed 2024-01-15 2024-01-15 14:00
      3            3     Monitor  349.99   shipped 2024-02-01 2024-02-02 08:00
      4            1       Mouse   25.00 completed 2024-02-10 2024-02-10 16:00
      5            4      Tablet  599.99 completed 2024-02-20 2024-02-20 11:00
      6            5  Headphones  199.99 completed 2024-03-01 2024-03-01 09:00
      7            2     USB Hub   49.99   pending 2024-03-05 2024-03-05 13:00
      8            6 Workstation 1299.99 completed 2024-03-10 2024-03-10 10:00
      9            7     Printer  459.99 completed 2024-04-01 2024-04-01 08:00
     10            8     Speaker  129.99   pending 2

,txn_id,customer_id,product,amount,status,created_at,updated_at
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00
8,9,7,Printer,459.99,completed,2024-04-01,2024-04-01 08:00
9,10,8,Speaker,129.99,pending,2024-04-02,2024-04-02 10:00


In [81]:
print("\n" + "="*60)
print("  SECTION 5 — CAPSTONE: MULTI-SOURCE PIPELINE")
print("="*60)


  SECTION 5 — CAPSTONE: MULTI-SOURCE PIPELINE


In [82]:
print("\nStep 1: Read files")
df_c = pd.read_csv(RAW / "customers.csv")


Step 1: Read files


In [83]:
df_c

,customer_id,name,email,city,tier
0,1,Alice Johnson,alice@email.com,New York,Gold
1,2,Bob Smith,bob@email.com,Chicago,Silver
2,3,Carol White,carol@email.com,Los Angeles,Gold
3,4,David Lee,david@email.com,Houston,Bronze
4,5,Eva Brown,eva@email.com,Phoenix,Silver


In [84]:
with open(RAW / "orders.json") as f:
    df_o = pd.DataFrame(json.load(f))

In [85]:
df_c["source"] = "file:customers"
df_o["source"] = "file:orders"

In [87]:
file_df = pd.concat([df_c, df_o], ignore_index=True)
print(f"  Files → {len(file_df)} rows")

  Files → 10 rows


In [88]:
file_df

,customer_id,name,email,city,tier,source,order_id,product,qty,price,status
0,1,Alice Johnson,alice@email.com,New York,Gold,file:customers,NaN,NaN,NaN,NaN,NaN
1,2,Bob Smith,bob@email.com,Chicago,Silver,file:customers,NaN,NaN,NaN,NaN,NaN
2,3,Carol White,carol@email.com,Los Angeles,Gold,file:customers,NaN,NaN,NaN,NaN,NaN
3,4,David Lee,david@email.com,Houston,Bronze,file:customers,NaN,NaN,NaN,NaN,NaN
4,5,Eva Brown,eva@email.com,Phoenix,Silver,file:customers,NaN,NaN,NaN,NaN,NaN
5,1,NaN,NaN,NaN,NaN,file:orders,101.0,Laptop,1.0,999.99,delivered
6,2,NaN,NaN,NaN,NaN,file:orders,102.0,Mouse,2.0,25.00,shipped
7,3,NaN,NaN,NaN,NaN,file:orders,103.0,Monitor,1.0,349.99,pending
8,1,NaN,NaN,NaN,NaN,file:orders,104.0,Keyboard,1.0,79.99,delivered
9,5,NaN,NaN,NaN,NaN,file:orders,105.0,Webcam,1.0,89.99,shipped


In [89]:
print("\nStep 2: Fetch from API")


Step 2: Fetch from API


In [90]:
try:
    raw_users = api_get("/users")
    api_df = pd.json_normalize(raw_users)[["id", "name", "email"]]
    api_df.columns = ["customer_id", "name", "email"]
    api_df["source"] = "api:users"
    print(f"  API → {len(api_df)} rows")
except Exception as e:
    print(f"  API failed ({e}), skipping.")
    api_df = pd.DataFrame()


  API → 10 rows


In [91]:
raw_users = api_get("/users")

In [93]:
pd.json_normalize(raw_users)[["id", "name", "email"]]

,id,name,email
0,1,Leanne Graham,Sincere@april.biz
1,2,Ervin Howell,Shanna@melissa.tv
2,3,Clementine Bauch,Nathan@yesenia.net
3,4,Patricia Lebsack,Julianne.OConner@kory.org
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me
8,9,Glenna Reichert,Chaim_McDermott@dana.io
9,10,Clementina DuBuque,Rey.Padberg@karina.biz


In [94]:
# Step 3 — Database (incremental)
print("\nStep 3: Database (incremental)")
db_df = pd.read_sql("SELECT * FROM transactions", conn)
db_df["source"] = "db:transactions"
print(f"  Database → {len(db_df)} rows")


Step 3: Database (incremental)
  Database → 11 rows


In [95]:
# Step 4 — Merge
print("\nStep 4: Merge all sources")
all_dfs = [df for df in [file_df, api_df, db_df] if not df.empty]
merged = pd.concat(all_dfs, ignore_index=True)
merged.drop_duplicates(inplace=True)
merged["ingested_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"  Merged → {len(merged)} rows, {len(merged.columns)} columns")


Step 4: Merge all sources
  Merged → 31 rows, 16 columns


In [96]:
merged

,customer_id,name,email,city,tier,source,order_id,product,qty,price,status,txn_id,amount,created_at,updated_at,ingested_at
0,1,Alice Johnson,alice@email.com,New York,Gold,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
1,2,Bob Smith,bob@email.com,Chicago,Silver,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
2,3,Carol White,carol@email.com,Los Angeles,Gold,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
3,4,David Lee,david@email.com,Houston,Bronze,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
4,5,Eva Brown,eva@email.com,Phoenix,Silver,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
5,1,NaN,NaN,NaN,NaN,file:orders,101.0,Laptop,1.0,999.99,delivered,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
6,2,NaN,NaN,NaN,NaN,file:orders,102.0,Mouse,2.0,25.00,shipped,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
7,3,NaN,NaN,NaN,NaN,file:orders,103.0,Monitor,1.0,349.99,pending,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
8,1,NaN,NaN,NaN,NaN,file:orders,104.0,Keyboard,1.0,79.99,delivered,NaN,NaN,NaN,NaN,2026-02-28 20:50:58
9,5,NaN,NaN,NaN,NaN,file:orders,105.0,Webcam,1.0,89.99,shipped,NaN,NaN,NaN,NaN,2026-02-28 20:50:58


In [97]:
print("\nStep 5: Save output")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out = PROCESSED / f"pipeline_output_{ts}.csv"
merged.to_csv(out, index=False)
print(f"  Saved → {out}")


Step 5: Save output
  Saved → data\processed\pipeline_output_20260228_205139.csv


In [98]:
print("\n" + "="*60)
print("  PIPELINE COMPLETE")
print("="*60)
print(f"  File rows     : {len(file_df)}")
print(f"  API rows      : {len(api_df)}")
print(f"  DB rows       : {len(db_df)}")
print(f"  Total merged  : {len(merged)}")
print(f"  Output        : {out.name}")


  PIPELINE COMPLETE
  File rows     : 10
  API rows      : 10
  DB rows       : 11
  Total merged  : 31
  Output        : pipeline_output_20260228_205139.csv


In [99]:
source_counts = merged["source"].value_counts().to_dict()
for src, cnt in source_counts.items():
    print(f"    {src}: {cnt} rows")


    db:transactions: 11 rows
    api:users: 10 rows
    file:orders: 5 rows
    file:customers: 5 rows


In [100]:
conn.close()